# 🧠 Exploratory Data Analysis (EDA) — Cyberbullying Detection Dataset

This notebook performs exploratory data analysis on the preprocessed dataset.  
We'll examine class balance, text characteristics, frequent words, and word clouds to understand the data better before model training.

Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from collections import Counter
import plotly.express as px

# Enable inline plotting
%matplotlib inline

# Optional: Set theme
sns.set(style="whitegrid")

Step 1: Load the Preprocessed Dataset

In [ ]:
# Load preprocessed data
df = pd.read_csv("../data/processed/preprocessed_data.csv")
print(f"✅ Dataset loaded successfully. Shape: {df.shape}")

# Display first few rows
df.head()

Step 2: Basic Dataset Info

In [ ]:
# Basic structure
df.info()

# Missing values check
print("\nMissing values per column:")
print(df.isnull().sum())

# Summary statistics for text lengths
df["clean_text"] = df["clean_text"].astype(str)  # ensure all values are strings
df["text_length"] = df["clean_text"].apply(lambda x: len(str(x)))
df["word_count"] = df["clean_text"].apply(lambda x: len(str(x).split()))
df[["text_length", "word_count"]].describe()



Step 3: Class Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(
    data=df,
    x="cyberbullying_type",
    hue="cyberbullying_type",   # same variable assigned to hue
    palette="viridis",
    legend=False
)
plt.title("Class Distribution of Cyberbullying Types")
plt.xlabel("Cyberbullying Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../results/eda_class_distribution.png", bbox_inches="tight")
plt.show()

Optional: Plotly version (interactive)

In [ ]:
cyberbullying_counts = (
    df["cyberbullying_type"]
    .value_counts()
    .reset_index()
)
cyberbullying_counts.columns = ["cyberbullying_type", "count"]

fig = px.bar(
    cyberbullying_counts,
    x="cyberbullying_type",
    y="count",
    title="Cyberbullying Type Distribution (Interactive)",
    labels={"cyberbullying_type": "Cyberbullying Type", "count": "Count"},
    color="cyberbullying_type",
)
fig.write_html("../results/eda_class_distribution_interactive.html")
fig.show()

Step 4: Text Length Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["word_count"], bins=40, kde=True, color='skyblue')
plt.title("Distribution of Word Count per Text")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("../results/eda_word_count_distribution.png", bbox_inches="tight")
plt.show()

Step 5: Top 20 Most Common Words

In [ ]:
# Most Common Words
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def get_top_n_words(texts, n=20):
    words = " ".join(texts).split()
    words = [w for w in words if w not in stop_words]
    most_common = Counter(words).most_common(n)
    return pd.DataFrame(most_common, columns=["word", "count"])

top_words = get_top_n_words(df["clean_text"])
plt.figure(figsize=(10,6))
sns.barplot(x="count", y="word", data=top_words, palette="magma")
plt.title("Top 20 Most Common Words in Dataset")
plt.tight_layout()
plt.savefig("../results/eda_top_words.png", bbox_inches="tight")
plt.show()

Step 6: Word Clouds by Class

In [ ]:
# WordClouds by Class
for label in df["cyberbullying_type"].unique():
    text = " ".join(df[df["cyberbullying_type"] == label]["clean_text"])
    wordcloud = WordCloud(width=800, height=400, background_color="white").generate(text)
    plt.figure(figsize=(10,5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud — {label}")
    plt.tight_layout()
    plt.savefig(f"../results/wordcloud_{label}.png", bbox_inches="tight")
    plt.show()

Step 7: Summary Statistics by Class

In [ ]:
summary = df.groupby("cyberbullying_type")[["text_length", "word_count"]].mean().round(1)
print("\n📊 Average text length and word count per category:")
display(summary)
summary.to_csv("../results/eda_summary_stats.csv")

Step 8: Correlation Matrix

In [ ]:
# Correlation
numeric_df = df.select_dtypes(include=["number"])
if not numeric_df.empty:
    plt.figure(figsize=(6,4))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.savefig("../results/eda_correlation_matrix.png", bbox_inches="tight")
    plt.show()